# SQL 4: CASE expressions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-04-case-expressions.ipynb)

`CASE` is SQL's if/else. It turns values into labels (bucketing), into flags, and, inside an aggregate, into
columns (a pivot). Once you can write `CASE`, most reporting logic fits in one query.

This chapter shows how `CASE` is evaluated, how it works with aggregates, and how Spark's `PIVOT` clause differs.
It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

### The data

- `customers` has 8 rows. Customers 7 and 8 have no orders.
- `orders` has 14 rows for customers 1 to 6: 11 `completed`, 2 `cancelled` (orders 5 and 14) and 1 `pending`
  (order 9). Amounts range from 100 to 500.

The check below confirms those counts, so later results can be worked out by hand.

In [ ]:
q = "SELECT status, COUNT(*) AS orders FROM orders GROUP BY status ORDER BY status"
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 2), ("completed", 11), ("pending", 1)]

## 1. The syntax

A `CASE` expression produces one value per row. A **searched** `CASE` lists conditions in order. The result is the
value of the **first** condition that is true; later branches are not checked for that row.

```sql
CASE
  WHEN condition1 THEN value1
  WHEN condition2 THEN value2
  ELSE default_value
END
```

If no condition is true and there is no `ELSE`, the result is `NULL`. The order of the branches is part of the
logic. All branches should return the same type, because the result is one column.

`CASE` is an expression, not a statement, so it can appear anywhere a value can: in `SELECT`, `WHERE`, `GROUP BY`,
`ORDER BY` and inside aggregate functions.

### Where you'll use it

- **Business rules in SQL:** a discount tier or a fraud flag written as one `CASE` in a model.
- **Cleaning source codes:** mapping `'M'`, `'F'` and `'U'` from a source system to readable labels.

Work it out: for the value 500, both `500 >= 100` and `500 >= 300` are true. The first one wins, so the result is
`'a'`. In the second column no condition is true and there is no `ELSE`, so the result is NULL.

In [ ]:
q = """
SELECT
  CASE WHEN 500 >= 100 THEN 'a' WHEN 500 >= 300 THEN 'b' END AS first_match_wins,
  CASE WHEN 1 = 2 THEN 'x' END                                AS no_else_gives_null
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("a", None)]

### Simple CASE

When every branch compares the same expression with `=`, a **simple** `CASE` is shorter. It names the expression once
and lists the values:

```sql
CASE status WHEN 'completed' THEN 1 WHEN 'pending' THEN 2 ELSE 3 END
```

This means exactly `CASE WHEN status = 'completed' THEN 1 WHEN status = 'pending' THEN 2 ELSE 3 END`. Because it uses
`=`, a simple `CASE` can never match NULL (section 5).

In [ ]:
simple = """
SELECT order_id, CASE status WHEN 'completed' THEN 1 WHEN 'pending' THEN 2 ELSE 3 END AS rank
FROM orders ORDER BY order_id
"""
searched = """
SELECT order_id, CASE WHEN status = 'completed' THEN 1 WHEN status = 'pending' THEN 2 ELSE 3 END AS rank
FROM orders ORDER BY order_id
"""
assert spark.sql(simple).collect() == spark.sql(searched).collect()
assert [r.rank for r in spark.sql(simple).collect()][:9] == [1, 1, 1, 1, 3, 1, 1, 1, 2]

### A branch runs only for rows that reach it

Because the first true branch stops the evaluation for that row, a `CASE` can guard an expression that would fail.
With ANSI mode on, dividing by zero raises an error. In the query below the row with `y = 0` takes the first branch,
so `10 / y` is never computed for it.

In [ ]:
q = """
SELECT y, CASE WHEN y = 0 THEN NULL ELSE 10 / y END AS ratio
FROM VALUES (0), (5) AS t(y)
ORDER BY y
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(0, None), (5, 2.0)]

## 2. Bucketing values

**Bucketing** means mapping a continuous value, such as an amount, onto a small set of labels. Label each order as
`small`, `medium` or `large`. The branches are checked from top to bottom, so an order of 300 is `large` even though it
also satisfies `>= 150`.

Because the branches are tested in order, each `WHEN` only needs a lower bound. A row reaches the `medium` branch only
if it already failed `>= 300`, so `medium` really means 150 to 299.

By hand for the first six orders: 100 is small, 200 medium, 150 medium, 300 large, 250 medium, 400 large.

### Where you'll use it

- **Age bands and price tiers** for segmentation dashboards.
- **Risk categories** from a score, such as low, medium and high.

In [ ]:
q = """
SELECT order_id, amount,
       CASE WHEN amount >= 300 THEN 'large'
            WHEN amount >= 150 THEN 'medium'
            ELSE 'small'
       END AS size
FROM orders
ORDER BY order_id
LIMIT 6
"""
spark.sql(q).show()
assert [r.size for r in spark.sql(q).collect()] == ["small", "medium", "medium", "large", "medium", "large"]

### Wrong branch order

Swap the first two branches and every large order is caught by `>= 150` first. Orders 4 (300) and 6 (400) become
`medium`, and the `large` branch can never be reached. Put the most restrictive condition first.

In [ ]:
q = """
SELECT order_id,
       CASE WHEN amount >= 150 THEN 'medium'
            WHEN amount >= 300 THEN 'large'
            ELSE 'small'
       END AS size
FROM orders
ORDER BY order_id
LIMIT 6
"""
assert [r.size for r in spark.sql(q).collect()] == ["small", "medium", "medium", "medium", "medium", "medium"]

## 3. CASE inside an aggregate

Put a `CASE` inside `COUNT` or `SUM` and each condition becomes its own column. This is called **conditional
aggregation**. `COUNT(CASE ... THEN 1 END)` counts only the matching rows, because a `CASE` with no `ELSE` gives NULL,
and `COUNT` ignores NULL. The `LEFT JOIN` keeps customers with no orders, and they show zeros.

Work it out for customer 2, whose orders are completed, cancelled and completed. The `completed` column's `CASE`
produces 1, NULL, 1, and `COUNT` gives 2. The `cancelled` column's `CASE` produces NULL, 1, NULL, and `COUNT` gives 1.
Customer 7 has one joined row in which every order column is NULL, so both `CASE`s give NULL and both counts are 0.

### Where you'll use it

- **Conditional metrics:** the number of refunded orders and the total refunded amount in the same `GROUP BY` query.
- **One-pass pivots:** counting each status as its own column without a second query.

In [ ]:
q = """
SELECT c.customer_id, c.name,
       COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
       COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert [(r.customer_id, r.completed, r.cancelled) for r in spark.sql(q).collect()] == [
    (1, 3, 0), (2, 2, 1), (3, 2, 0), (4, 1, 0), (5, 2, 0), (6, 1, 1), (7, 0, 0), (8, 0, 0),
]

### The ELSE 0 trap in COUNT

`COUNT` counts every non-NULL value, and 0 is not NULL. So `COUNT(CASE WHEN ... THEN 1 ELSE 0 END)` counts **every**
row, matching or not. Customer 2 gets 3 instead of 2, and customer 7, who has no orders at all, gets 1 (the single
NULL row from the `LEFT JOIN` falls into `ELSE 0`). With `COUNT`, leave out the `ELSE`.

In [ ]:
q = """
SELECT c.customer_id,
       COUNT(CASE WHEN o.status = 'completed' THEN 1 ELSE 0 END) AS wrong_completed
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY c.customer_id
"""
assert spark.sql(q).collect() == [(1, 3), (2, 3), (3, 2), (4, 2), (5, 2), (6, 2), (7, 1), (8, 1)]

### SUM with CASE

To total an amount for matching rows only, return the amount from the `THEN` branch and sum. Here `ELSE 0` is what
you want: a group with no matching rows sums to 0. Without `ELSE`, every value in such a group is NULL, and the sum of
only NULLs is NULL.

By hand for customer 2: completed amounts 300 + 400 = 700; cancelled amount 250. Customer 1 has no cancelled order,
so its cancelled total is 0 with `ELSE 0` and NULL without it.

In [ ]:
q = """
SELECT c.customer_id,
       SUM(CASE WHEN o.status = 'completed' THEN o.amount ELSE 0 END) AS completed_revenue,
       SUM(CASE WHEN o.status = 'cancelled' THEN o.amount ELSE 0 END) AS cancelled_amount,
       SUM(CASE WHEN o.status = 'cancelled' THEN o.amount END)        AS cancelled_no_else
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, 450, 0, None), (2, 700, 250, 250), (3, 300, 0, None), (4, 320, 0, None),
    (5, 950, 0, None), (6, 275, 325, 325), (7, 0, 0, None), (8, 0, 0, None),
]

### The FILTER clause

Spark and PostgreSQL also accept `FILTER (WHERE ...)` after an aggregate. It means the same as putting a `CASE` inside
the aggregate, and reads more directly. `CASE` works on every engine, so it remains the portable form.

In [ ]:
with_filter = """
SELECT c.customer_id, COUNT(o.order_id) FILTER (WHERE o.status = 'completed') AS completed
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id ORDER BY c.customer_id
"""
with_case = """
SELECT c.customer_id, COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id ORDER BY c.customer_id
"""
assert spark.sql(with_filter).collect() == spark.sql(with_case).collect()

## 4. The PIVOT clause

Spark also has a `PIVOT` clause that creates the columns from the values of one column. It is shorter, but two
behaviours matter. Combinations that never occur become NULL, not zero. And a group with no matching rows at all,
such as customers 7 and 8, is missing from the result.

`PIVOT` has no `GROUP BY`. It groups by **every input column that is not aggregated and not named after `FOR`**. The
subquery below keeps only `customer_id` and `status`, so the result has one row per customer.

### Where you'll use it

- **Cross-tab reports:** monthly revenue by product as columns, the usual shape of finance packs. Spark and Snowflake offer `PIVOT` directly.

In [ ]:
q = """
SELECT * FROM (SELECT customer_id, status FROM orders)
PIVOT (COUNT(*) FOR status IN ('completed' AS completed, 'cancelled' AS cancelled))
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, 3, None), (2, 2, 1), (3, 2, None), (4, 1, None), (5, 2, None), (6, 1, 1),
]

### Extra columns change the grouping

If the subquery also keeps `order_id`, the implicit grouping becomes (`customer_id`, `order_id`), and the pivot
returns one row per order instead of one per customer. Select only the columns you want to group by.

In [ ]:
q = """
SELECT * FROM (SELECT customer_id, order_id, status FROM orders)
PIVOT (COUNT(*) FOR status IN ('completed' AS completed, 'cancelled' AS cancelled))
"""
assert spark.sql(q).count() == 14

### Turning the NULLs into zeros

Wrap the pivoted columns in `COALESCE` to show 0 for missing combinations. Customers 7 and 8 are still absent; to
include them, use the conditional aggregation from section 3 with a `LEFT JOIN` from `customers`.

In [ ]:
q = """
SELECT customer_id, COALESCE(completed, 0) AS completed, COALESCE(cancelled, 0) AS cancelled
FROM (SELECT customer_id, status FROM orders)
PIVOT (COUNT(*) FOR status IN ('completed' AS completed, 'cancelled' AS cancelled))
ORDER BY customer_id
"""
assert spark.sql(q).collect() == [(1, 3, 0), (2, 2, 1), (3, 2, 0), (4, 1, 0), (5, 2, 0), (6, 1, 1)]

## 5. CASE and NULL

A `WHEN` condition that compares with NULL is never true, the same trap as in `WHERE`. Test for missing values with
`IS NULL`. A condition that is unknown is treated like false: the row moves on to the next branch.

### Where you'll use it

- **Safe labels:** a `CASE` with an `ELSE` branch makes sure an unexpected value is visible instead of becoming NULL.
- **Explicit missing labels:** `WHEN region IS NULL THEN 'unknown'` as the first branch, so missing values get their own bucket.

In [ ]:
q = """
SELECT
  CASE WHEN CAST(NULL AS INT) = NULL THEN 'match' ELSE 'no match' END AS equals_null,
  CASE WHEN CAST(NULL AS INT) IS NULL THEN 'match' ELSE 'no match' END AS is_null
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("no match", "match")]

### Simple CASE cannot match NULL

`CASE email WHEN NULL THEN ...` is shorthand for `email = NULL`, which is never true. So the missing email below is
labelled `present`. The searched form with `IS NULL` labels it correctly.

In [ ]:
q = """
SELECT id,
       CASE email WHEN NULL THEN 'missing' ELSE 'present' END        AS simple_form,
       CASE WHEN email IS NULL THEN 'missing' ELSE 'present' END     AS searched_form
FROM VALUES (1, 'a@x.com'), (2, NULL) AS t(id, email)
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "present", "present"), (2, "present", "missing")]

### Custom sort order

`ORDER BY` sorts by the value of an expression, so a `CASE` can impose any order you need. Here statuses sort as
completed, pending, cancelled, instead of alphabetically. The simple `CASE` maps each status to a number, and the
rows sort by that number.

### Where you'll use it

- **Business ordering:** sorting statuses as "new, in progress, done" rather than alphabetically.

In [ ]:
q = """
SELECT DISTINCT status
FROM orders
ORDER BY CASE status WHEN 'completed' THEN 1 WHEN 'pending' THEN 2 ELSE 3 END
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("completed",), ("pending",), ("cancelled",)]

## Interview notes

- **Order of WHEN matters.** The first true branch wins, so put the most specific conditions first.
- **CASE in aggregates** is the portable way to pivot. `PIVOT` is shorter where it exists, but it does not fill
  missing combinations with zero, and it drops groups with no rows.
- **COUNT versus SUM.** Inside `COUNT`, leave out `ELSE` (or use `ELSE NULL`); `ELSE 0` counts every row. Inside `SUM`,
  `ELSE 0` makes an empty group total 0 instead of NULL.
- **NULL in CASE.** An unknown condition falls through to the next branch. A simple `CASE` cannot match NULL; use
  `WHEN x IS NULL`.
- **CASE vs IF.** `CASE` is standard SQL and works across engines. `IF` (Spark, MySQL) and `IIF` (SQL Server) exist in
  some engines only.

## Summary

- A `CASE` returns the first matching branch. Order the conditions from the most specific to the least specific.
- Without an `ELSE`, an unmatched row gets NULL.
- A simple `CASE` is shorthand for a chain of `=` comparisons on one expression.
- Conditional aggregation (`COUNT(CASE ...)`, `SUM(CASE ... ELSE 0 END)`) is portable. `PIVOT` is shorter, but it
  produces NULL for missing combinations.

## Practice questions

1. Count the orders in each size label, where `large` means an amount of 300 or more and `small` means less than 300.
   Sort by label.
2. For each customer with orders, show the number of pending orders and the number of completed orders. Sort by `customer_id`.
3. Label each customer `has cancelled` or `no cancelled`, depending on whether they have any cancelled order.
   Show `customer_id` and the label for customers with orders, sorted by `customer_id`.
4. Count the orders in each of three labels: `large` (300 or more), `medium` (150 to 299) and `small` (under 150).
   Sort the labels as large, medium, small.
5. For every region, show the completed revenue and the number of orders that are not completed. Include regions
   without orders with zeros, and sort by region.

## Answers

### Question 1

Bucket with a `CASE` and group by the same expression. Amounts of 300 or more: 300, 400, 320, 500, 450 and 325, so 6
large and 8 small.

In [ ]:
q = """
SELECT CASE WHEN amount >= 300 THEN 'large' ELSE 'small' END AS size, COUNT(*) AS orders
FROM orders
GROUP BY CASE WHEN amount >= 300 THEN 'large' ELSE 'small' END
ORDER BY size
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("large", 6), ("small", 8)]

### Question 2

Conditional aggregation: one `COUNT(CASE ...)` per status, grouped by customer.

In [ ]:
q = """
SELECT customer_id,
       COUNT(CASE WHEN status = 'pending' THEN 1 END)   AS pending,
       COUNT(CASE WHEN status = 'completed' THEN 1 END) AS completed
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 0, 3), (2, 0, 2), (3, 0, 2), (4, 1, 1), (5, 0, 2), (6, 0, 1)]

### Question 3

Count the cancelled orders per customer, then wrap that aggregate in an outer `CASE` that turns the count into a label.

In [ ]:
q = """
SELECT customer_id,
       CASE WHEN COUNT(CASE WHEN status = 'cancelled' THEN 1 END) > 0
            THEN 'has cancelled' ELSE 'no cancelled' END AS label
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, "no cancelled"), (2, "has cancelled"), (3, "no cancelled"),
    (4, "no cancelled"), (5, "no cancelled"), (6, "has cancelled"),
]

### Question 4

Bucket in a subquery with the most restrictive branch first, then sort with a second `CASE`. Medium amounts are 200,
150, 250, 180, 220 and 275; small amounts are 100 and 120.

In [ ]:
q = """
SELECT size, COUNT(*) AS orders
FROM (
  SELECT CASE WHEN amount >= 300 THEN 'large'
              WHEN amount >= 150 THEN 'medium'
              ELSE 'small' END AS size
  FROM orders
)
GROUP BY size
ORDER BY CASE size WHEN 'large' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("large", 6), ("medium", 6), ("small", 2)]

### Question 5

Start from `customers` with a `LEFT JOIN` so the East region (customers 7 and 8) survives. Use `SUM(... ELSE 0)` for
revenue and `COUNT` without `ELSE` for the order count. North: 450 + 700 = 1,150 and one cancelled order.

In [ ]:
q = """
SELECT c.region,
       SUM(CASE WHEN o.status = 'completed' THEN o.amount ELSE 0 END)       AS completed_revenue,
       COUNT(CASE WHEN o.status IN ('pending', 'cancelled') THEN 1 END)     AS not_completed
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.region
ORDER BY c.region
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("East", 0, 0), ("North", 1150, 1), ("South", 620, 1), ("West", 1225, 1)]